# LuxeForLess VTO on Kaggle GPU

1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → ON**
3. Kaggle **Secrets**: `NGROK_AUTHTOKEN` from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
4. GitHub repo must be **public** (or add optional secret `GITHUB_TOKEN` for private repos)

When done, copy the **ngrok HTTPS URL** into Vercel → `NEXT_PUBLIC_VTO_SERVICE_URL`

> **Note:** Kaggle does NOT auto-update from GitHub. Re-run this notebook after code changes. Vercel auto-deploys only if GitHub is connected in Vercel dashboard.

In [ ]:
# ONE CELL — run after every Kaggle session restart (/kaggle/working is wiped)
import os, sys, shutil, subprocess, time

WORK = "/kaggle/working/luxeforless"
SCRIPT = f"{WORK}/deploy/kaggle/run_vto.py"
REPO = "https://github.com/mintukumar0000/luxeforless.git"
NGROK_TOKEN_FALLBACK = ""  # only if Add-ons → Secrets fails

print("Step 0: starting...")

# /kaggle/working is cleared when session ends — clone OR pull
if os.path.exists(WORK) and not os.path.exists(SCRIPT):
    print("Removing broken clone...")
    shutil.rmtree(WORK)

if not os.path.exists(SCRIPT):
    print("Cloning repo (session was cleared — normal)...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, WORK])
else:
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "-C", WORK, "pull"], check=False)

print("Killing old server...")
subprocess.run(["pkill", "-9", "-f", "uvicorn"], check=False)
time.sleep(2)

ngrok_token = os.environ.get("NGROK_AUTHTOKEN", "")
if not ngrok_token:
    try:
        from kaggle_secrets import UserSecretsClient
        ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    except Exception as e:
        print(f"Kaggle Secrets unavailable ({e})")
        print("Fix: Add-ons → Secrets → NGROK_AUTHTOKEN, or use NGROK_TOKEN_FALLBACK.")
if not ngrok_token and NGROK_TOKEN_FALLBACK:
    ngrok_token = NGROK_TOKEN_FALLBACK
if not ngrok_token:
    raise RuntimeError("NGROK_AUTHTOKEN missing — add secret or set NGROK_TOKEN_FALLBACK")

os.environ["NGROK_AUTHTOKEN"] = ngrok_token
os.environ["LUXEFORLESS_REPO_URL"] = REPO
os.environ["VTO_NUM_TIMESTEPS"] = "20"
print("NGROK OK")

print("\nRunning bootstrap (~10-15 min first time, includes AI pre-load)...")
print("Wait for: AI pipeline ready + VTO PUBLIC URL\n")
subprocess.check_call([sys.executable, SCRIPT])

In [ ]:
import os
import shutil
import subprocess
import sys

# Self-contained: works even if Cell 1 wasn't run (e.g. after session restart)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ.setdefault("NGROK_AUTHTOKEN", secrets.get_secret("NGROK_AUTHTOKEN"))
except Exception:
    pass

os.environ.setdefault("LUXEFORLESS_REPO_URL", "https://github.com/mintukumar0000/luxeforless.git")
os.environ.setdefault("VTO_NUM_TIMESTEPS", "4")

if not os.environ.get("NGROK_AUTHTOKEN"):
    raise RuntimeError("NGROK_AUTHTOKEN missing — add it in Add-ons → Secrets, then run this cell again")

work = "/kaggle/working/luxeforless"
script = f"{work}/deploy/kaggle/run_vto.py"
repo = os.environ["LUXEFORLESS_REPO_URL"]

# Optional: private repo support via Kaggle secret GITHUB_TOKEN
try:
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if token and "github.com" in repo and "@" not in repo:
        repo = repo.replace("https://", f"https://{token}@")
        print("Using GITHUB_TOKEN for private repo clone")
except Exception:
    pass

if os.path.exists(work) and not os.path.exists(script):
    print("Removing incomplete clone...")
    shutil.rmtree(work)

if not os.path.exists(script):
    print("Cloning luxeforless repo...")
    subprocess.check_call(["git", "clone", "--depth", "1", repo, work])

print("NGROK token loaded: yes")
print("Repo:", repo)
print("Starting VTO service (first run downloads ~2GB weights — allow 10-15 min)...")
print("When you see the ngrok HTTPS URL below, paste it into Vercel as NEXT_PUBLIC_VTO_SERVICE_URL\n")

subprocess.check_call([sys.executable, script])